# Adversarial TTS - Class-Based Architecture

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles model loading and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

## 1. Imports

In [ ]:
import torch
import os
import argparse
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt_tab')

os.chdir("..")  # Since we are in Scripts Folder

from Datastructures.dataclass import ModelData
from Objectives.FitnessObjective import FitnessObjective

# Import class-based modules
from Trainer.EnvironmentLoader import EnvironmentLoader
from Trainer.AdversarialTrainer import AdversarialTrainer
from Trainer.RunLogger import RunLogger
from Trainer.VectorManipulator import VectorManipulator

# Import Pymoo components
from Optimizer.pymoo_optimizer import PymooOptimizer
from pymoo.algorithms.moo.nsga2 import NSGA2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

## 2. Configuration

Edit the values below to configure the optimization run.

In [4]:
# =============================================================================
# Configuration - Edit these values directly
# =============================================================================
args = argparse.Namespace(
    # Text parameters
    ground_truth_text="I think the NFL is lame and boring",
    target_text="The Seattle Seahawks are the best Team in the world",

    # Optimization parameters
    loop_count=1,
    num_generations=1,
    pop_size=1,
    iv_scalar=0.5,
    size_per_phoneme=1,
    batch_size=100,  # -1 for full batch

    # Flags
    notify=False,
    subspace_optimization=False,

    # Mode and objectives
    mode="TARGETED",  # TARGETED, UNTARGETED, NOISE_UNTARGETED
    objectives="PER_TARGET=0.0",
)

## 3. Initialize Environment

Load models and prepare audio data.

In [5]:
# Initialize environment using EnvironmentLoader
loader = EnvironmentLoader(device)
config_data = loader.load_configuration(args)
config_data.print_summary()

tts_model, asr_model = loader.load_required_models()

audio_gt, audio_target, audio_embedding_gt, audio_embedding_target = loader.generate_audio_data(
    config_data.mode, config_data.text_gt, config_data.text_target, tts_model
)

print("\nEnvironment initialized successfully!")

=== Configuration ===
GT Text:               I think the NFL is lame and boring
Target Text:           The Seattle Seahawks are the best Team in the world
Generations:           1
Population Size:       1
Loop Count:            1
IV Scalar:             0.5
Size Per Phoneme:      1
Batch Size:            1
Subspace Optimization: False
Notify (WhatsApp):     False
Mode:                  TARGETED
Objectives:            ['PER_TARGET']
Thresholds:            PER_TARGET<=0.0
Cuda on (1 GPUs)
Loading TTS Model (StyleTTS2)...
Loading ASR Model (Whisper)...

Environment initialized successfully!


## 4. Setup Trainer Components

Initialize Objectives, VectorManipulator, Trainer, and Logger.

In [6]:
# Initialize Objectives
objectives = loader.initialize_objectives(
    active_objectives=config_data.active_objectives,
    model_data=ModelData(tts_model=tts_model, asr_model=asr_model),
    text_gt=config_data.text_gt,
    text_target=config_data.text_target,
    mode=config_data.mode,
    audio_gt=audio_gt,
)

vector_manipulator = VectorManipulator(audio_embedding_gt, audio_embedding_target.h_text, config_data)

# Create Trainer and Logger
trainer = AdversarialTrainer(
    tts_model, asr_model, config_data.thresholds, objectives, vector_manipulator, device
)
logger = RunLogger(
    config_data.active_objectives, tts_model, asr_model, vector_manipulator, device
)

print("Trainer components initialized!")

Initialized PER_TARGET (batching=True)
Trainer components initialized!


## 5. Run Optimization

Run the optimization loop and log results.

In [ ]:
for loop_iteration in range(config_data.loop_count):
    print(f"\n[Loop {loop_iteration + 1}/{config_data.loop_count}] Starting optimization loop...")

    # Initialize fresh optimizer for this cycle
    optimizer = PymooOptimizer(
        bounds=(0, 1),
        algorithm=NSGA2,
        algo_params={"pop_size": config_data.pop_size},
        num_objectives=len(config_data.active_objectives),
        solution_shape=(audio_embedding_gt.input_length.detach().cpu().item(), config_data.size_per_phoneme),
    )

    fitness_data, generation_count, elapsed_time_total = trainer.run_full_iteration(optimizer, config_data.num_generations, config_data.pop_size, config_data.batch_size)

    # 5. Save all results (audios, spectrograms, graphs, etc.)
    folder_path, text_best, best_candidate, audio_best = logger.save_all_results(
        optimizer, fitness_data, generation_count, elapsed_time_total,
        audio_gt, audio_target, config_data
    )

    print("[Log] Finished saving all results")